In [ ]:
import pandas as pd
import sqlite3
import os

In [ ]:
billing = pd.read_csv("../data/billing.csv")
patients = pd.read_csv("../data/patients.csv")
visits = pd.read_csv("../data/visits.csv")

print("Billing shape:", billing.shape)
print("Patients shape:", patients.shape)
print("visits shape:", visits.shape)

Billing shape: (25000, 7)
Patients shape: (5000, 7)
visits shape: (25000, 8)


In [ ]:
#create sqlite db
#create db folder if it doesnt exists

conn = sqlite3.connect("../db/hospital.db")
cursor = conn.cursor()

print("Database connection established:", conn)

Database connection established: <sqlite3.Connection object at 0x0000019BD044E110>


In [ ]:
billing.to_sql(
    name='billings',       # Name of the SQL table
    con=conn,              # Database connection object
    if_exists='replace',   # What to do if the table already exists ('fail', 'replace', or 'append')
    index=False            # Set to False to avoid writing the DataFrame index as a column
)
patients.to_sql(
    name='patients',       # Name of the SQL table
    con=conn,              # Database connection object
    if_exists='replace',   # What to do if the table already exists ('fail', 'replace', or 'append')
    index=False            # Set to False to avoid writing the DataFrame index as a column
)
visits.to_sql(
    name='visits',       # Name of the SQL table
    con=conn,              # Database connection object
    if_exists='replace',   # What to do if the table already exists ('fail', 'replace', or 'append')
    index=False            # Set to False to avoid writing the DataFrame index as a column
)

print("All three tables have been loaded")

All three tables have been loaded


In [ ]:
print("===first 3 rows of patients===")
print(pd.read_sql("select * from patients limit 3", conn).to_string())
print("===first 3 rows of visits===")
print(pd.read_sql("select * from visits limit 3", conn).to_string())
print("===first 3 rows of billings===")
print(pd.read_sql("select * from billings limit 3", conn).to_string())

===first 3 rows of patients===
   patient_id  age gender       city insurance_provider  chronic_flag registration_date
0           1   52      M  Bangalore            CareOne             1          5/6/2025
1           2   15      F     Mumbai            CareOne             0        12/27/2025
2           3   72      F  Bangalore         SecureLife             1         1/28/2025
===first 3 rows of visits===
   visit_id  patient_id  visit_date  department visit_type  length_of_stay_hours risk_score  doctor_id
0         1           1  10/16/2025  Cardiology        OPD                 23.15        Low        186
1         2           1   12/9/2025         ICU        ICU                 39.98       High        149
2         3           1  10/29/2025          ER         ER                 20.97     Medium        122
===first 3 rows of billings===
   bill_id  visit_id  billed_amount  approved_amount claim_status  payment_days billing_date
0        1         1       23550.00         21429.62

In [ ]:
#department workload analysis

dept_workload = pd.read_sql("""
SELECT
    department,
    count(visit_id)   AS total_visits,
    ROUND(AVG(length_of_stay_hours))  AS avg_los_hours,
    ROUND(MAX(length_of_stay_hours))  AS max_los_hours,
    SUM(CASE WHEN risk_score = 'High'
        THEN 1 ELSE 0 END)           AS high_risk_visits
    FROM visits
    GROUP BY department
    ORDER BY total_visits DESC
""" , conn)

print(dept_workload.to_string( index = False))


 department  total_visits  avg_los_hours  max_los_hours  high_risk_visits
    General          5757           14.0           37.0                24
Orthopedics          4474           23.0           53.0                54
 Cardiology          4071           33.0           71.0               489
         ER          3896           12.0           31.0               139
        ICU          3415           57.0           78.0              2975
  Neurology          3387           28.0           66.0               635


In [ ]:
#doctor risk cases
#which doctor handle the most high risk patients?

doctor_risk = pd.read_sql("""SELECT 
                            doctor_id,
                            COUNT(visit_id)                                 AS total_visits,
                            SUM(CASE WHEN risk_score = 'High'
                             THEN 1 else 0 END)                             AS high_risk_cases,
                            ROUND(
                                100.0 * SUM(CASE WHEN risk_score = 'High' 
                                then 1 else 0 end) / COUNT(visit_id), 1)    AS high_risk_pct
                            FROM visits
                            GROUP BY doctor_id
                            ORDER BY high_risk_cases DESC
                            LIMIT 10
                            
                        """, conn)

print("top 10 doctor by high risk cases")
print(doctor_risk.to_string(index=False))

top 10 doctor by high risk cases
 doctor_id  total_visits  high_risk_cases  high_risk_pct
       114           260               61           23.5
       130           264               60           22.7
       187           262               57           21.8
       126           244               55           22.5
       102           271               55           20.3
       189           240               54           22.5
       158           252               53           21.0
       171           260               52           20.0
       150           252               52           20.6
       194           267               51           19.1


In [ ]:
# patients visit pattern
#how many visits does a patient make on an average

visit_patterns = pd.read_sql("""
    SELECT 
        visit_frequency_bucket,
        COUNT(patient_id) AS num_patients
    FROM (
        SELECT
            patient_id,
            COUNT(visit_id) AS visit_count,
            CASE
                when COUNT(visit_id) =  1 THEN '1 visit' 
                when COUNT(visit_id) between 2 and 3 THEN '2-3 visits' 
                when COUNT(visit_id) between 4 and 5 THEN '4-5 visits'
                ELSE '6+ visits' 
            END AS visit_frequency_bucket
        FROM visits
        GROUP BY patient_id
    )
    GROUP BY visit_frequency_bucket
    ORDER BY num_patients DESC
                
""", conn)

print("Patients Visit frequency distribution")
print(visit_patterns.to_string(index=False))

Patients Visit frequency distribution
visit_frequency_bucket  num_patients
             6+ visits          1816
            4-5 visits          1776
            2-3 visits          1267
               1 visit           141


In [ ]:
risk_by_dept = pd.read_sql("""
    SELECT 
        department,
            SUM(CASE when risk_score = 'Low' Then 1 else 0 END) AS low,
            SUM(CASE when risk_score = 'Medium' Then 1 else 0 END) AS medium,
            SUM(CASE when risk_score = 'High' Then 1 else 0 END) AS high,
            COUNT(*) AS total
        FROM visits
        Group by department
        Order by high DESC                                  
                        
""", conn)

print("Risk score distribution by department")
print(risk_by_dept.to_string(index=False))


Risk score distribution by department
 department  low  medium  high  total
        ICU   10     430  2975   3415
  Neurology  875    1877   635   3387
 Cardiology 1063    2519   489   4071
         ER 1771    1986   139   3896
Orthopedics 3426     994    54   4474
    General 4590    1143    24   5757


In [ ]:
insurance_billing = pd.read_sql("""
    select 
        p.insurance_provider,
        COUNT(b.bill_id)                    AS total_claims,
        ROUND(SUM(b.billed_amount), 0)      AS total_billed,
        ROUND(AVG(b.billed_amount), 0)      AS avg_billed,
        ROUND(SUM(b.approved_amount), 0)    AS total_approved,
        SUM(CASE WHEN b.claim_status = 'Paid'
            THEN 1 ELSE 0 END)              AS paid,
        SUM(CASE WHEN b.claim_status = 'Pending'
            THEN 1 ELSE 0 END)              AS pending,
        SUM(CASE WHEN b.claim_status = 'Rejected'
            THEN 1 ELSE 0 END)              AS rejected
    FROM billing b
    JOIN visits v ON b.visit_id = v.visit_id
    JOIN patients p ON v.patient_id = p.patient_id
    GROUP BY p.insurance_provider
    ORDER BY total_billed DESC        
                                
                                """, conn)

print("Insurance billing breakdown")
print(insurance_billing.to_string(index=False))

Insurance billing breakdown
insurance_provider  total_claims  total_billed  avg_billed  total_approved  paid  pending  rejected
        HealthPlus          6499   135880246.0     20908.0     100998252.0  3907     1594       998
           CareOne          6326   133561604.0     21113.0      99210536.0  3766     1630       930
        SecureLife          6197   128382723.0     20717.0      94776267.0  3655     1558       984
         MediCareX          5978   123944363.0     20733.0      92170833.0  3612     1481       885


In [ ]:
#claim rejection analysis

rejection_analysis = pd.read_sql("""
    SELECT
        p.insurance_provider,
        COUNT(b.bill_id)                            AS total_claims,
        SUM(CASE WHEN b.claim_status = 'Rejected'  
            THEN 1 else 0 end)                      AS rejected_claims,
        ROUND(100.0 * sum(case when b.claim_status = 'Rejected'
            Then 1 else 0 end) / COUNT(b.bill_id), 1) AS rejection_rate_pct
    FROM billing b
    JOIN visits v ON b.visit_id = v.visit_id
    JOIN patients p ON v.patient_id = p.patient_id
    GROUP BY p.insurance_provider
    ORDER BY rejection_rate_pct DESC
            
                                 """, conn)

print("Claim rejetion rate by insurance provider")
print(rejection_analysis.to_string(index=False))

Claim rejetion rate by insurance provider
insurance_provider  total_claims  rejected_claims  rejection_rate_pct
        SecureLife          6197              984                15.9
        HealthPlus          6499              998                15.4
         MediCareX          5978              885                14.8
           CareOne          6326              930                14.7


In [ ]:
#revenue realization
#How much of what we bill is actually gets approved?

revenue_realization = pd.read_sql("""
    SELECT
        p.insurance_provider,
        ROUND(SUM(b.billed_amount), 0)      AS total_billed,
        ROUND(SUM(b.approved_amount), 0)    AS total_approved,
        ROUND(
            100.0*SUM(b.approved_amount)/
            SUM(b.billed_amount),1
        )                                   AS realization_rate_pct
        FROM billing b
        JOIN visits v ON b.visit_id = v.visit_id
        JOIN patients p ON v.patient_id = p.patient_id
        WHERE b.approved_amount IS NOT NULL
        GROUP BY p.insurance_provider
        ORDER BY realization_rate_pct DESC
                                  """, conn)
print("Revenue realization rate by Isurance provider")
print(revenue_realization.to_string(index=False))

Revenue realization rate by Isurance provider
insurance_provider  total_billed  total_approved  realization_rate_pct
           CareOne   126051757.0      99210536.0                  78.7
         MediCareX   117385192.0      92170833.0                  78.5
        HealthPlus   129030875.0     100998252.0                  78.3
        SecureLife   122301318.0      94776267.0                  77.5


In [ ]:
invalid_payment = pd.read_sql("""
    SELECT
        COUNT(CASE WHEN payment_days < 0 THEN 1 END) AS negative_days,
        COUNT(CASE WHEN payment_days > 365 THEN 1 END) AS over_1_year,
        COUNT(CASE WHEN payment_days IS NULL THEN 1 END) AS nulls, 
        MIN(payment_days)                               AS min_days,
        MAX(payment_days)                               AS max_days
    FROM billing                  
                              """, conn)

print("Payment days validation")
print(invalid_payment.to_string(index=False))

Payment days validation
 negative_days  over_1_year  nulls  min_days  max_days
             0            0    790       1.0      55.0


In [ ]:
model_table = pd.read_sql("""
    SELECT
        -- Patient features
        p.patient_id,
        p.age,
        p.gender,
        p.city,
        p.insurance_provider,
        p.chronic_flag,
        p.registration_date,

        -- Visit features
        v.visit_id,
        v.visit_date,
        v.department,
        v.visit_type,
        v.length_of_stay_hours,
        v.risk_score,
        v.doctor_id,

        -- Billing features
        b.bill_id,
        b.billed_amount,
        b.approved_amount,
        b.claim_status,
        b.payment_days,
        b.billing_date

    FROM visits v
    JOIN patients p ON v.patient_id = p.patient_id
    JOIN billing  b ON v.visit_id   = b.visit_id
    ORDER BY v.visit_date
""", conn)

print("model_table shape:", model_table.shape)
print("Columns:", model_table.columns.tolist())
model_table.head(3)

model_table shape: (25000, 20)
Columns: ['patient_id', 'age', 'gender', 'city', 'insurance_provider', 'chronic_flag', 'registration_date', 'visit_id', 'visit_date', 'department', 'visit_type', 'length_of_stay_hours', 'risk_score', 'doctor_id', 'bill_id', 'billed_amount', 'approved_amount', 'claim_status', 'payment_days', 'billing_date']


,patient_id,age,gender,city,insurance_provider,chronic_flag,registration_date,visit_id,visit_date,department,visit_type,length_of_stay_hours,risk_score,doctor_id,bill_id,billed_amount,approved_amount,claim_status,payment_days,billing_date
0,2,15,F,Mumbai,CareOne,0,12/27/2025,8,1/1/2026,General,OPD,9.63,Low,105,8,38178.73,38178.73,Paid,6.0,2025-06-27
1,12,3,M,Bangalore,CareOne,0,8/13/2025,65,1/1/2026,ICU,ICU,59.60,High,112,65,31990.78,21941.52,Pending,2.0,2025-10-02
2,129,44,M,Pune,MediCareX,1,7/20/2025,651,1/1/2026,ICU,ER,59.28,High,150,651,40040.66,20259.11,Pending,19.0,2026-01-18


In [ ]:
import os
os.makedirs("../outputs", exist_ok=True)
model_table.to_csv("../outputs/model_table.csv", index=False)
print(f"model_table.csv saved --> /outputs/model_table.csv")
print(f"Shape: {model_table.shape}")
print(f"Size: {os.path.getsize('../outputs/model_table.csv')/1024:.1f} KB")

model_table.csv saved --> /outputs/model_table.csv
Shape: (25000, 20)
Size: 3091.6 KB


In [ ]:
conn.close()
print("Database connection closed successfully")

Database connection closed successfully
